# 07 — Baseline Model Training

**FYP:** Developing a Semi-Dynamic Landslide Susceptibility Decision-Support Dashboard Using Machine Learning for Malaysia

This notebook trains the first two planned models:

1. **Logistic Regression** — interpretable baseline
2. **Random Forest** — primary nonlinear model

Both models use:

- the same `static_model_1to2.csv`
- the same predictors
- the same spatial validation groups
- the same 5 spatial folds

---

# Important interpretation

The target is:

- `1` = observed landslide location
- `0` = pseudo-absence / background control

Because the negative class is pseudo-absence rather than verified never-landslide data, the model output should primarily be interpreted as a:

> **relative landslide susceptibility score**

rather than a perfectly calibrated real-world probability that a landslide will occur.

Temporal/dynamic occurrence modelling is a later comparison.

---

# Primary evaluation metrics

- Accuracy
- Balanced Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- PR-AUC
- Confusion Matrix

For this FYP, pay particular attention to:

- **Balanced Accuracy**
- **Recall**
- **F1**
- **ROC-AUC**
- **PR-AUC**

and to consistency across spatial folds.


In [3]:
%pip install scikit-learn pandas numpy matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# OPTIONAL installation
# %pip install scikit-learn pandas numpy matplotlib

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import (
    GroupKFold,
    StratifiedGroupKFold,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

warnings.filterwarnings("ignore")

print("pandas:", pd.__version__)


pandas: 3.0.2


## 1. Project paths

In [5]:
PROJECT_ROOT = Path(r"C:\Landslide_FYP")

DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"

MODEL_READY_DIR = (
    DEV_DIR
    / "04_model_ready"
)

INPUT_PATH = (
    MODEL_READY_DIR
    / "static_model_1to2.csv"
)

OUTPUT_DIR = (
    DEV_DIR
    / "05_model_results"
    / "baseline_static"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Model-ready dataset not found:\\n{INPUT_PATH}"
    )

print("Input:", INPUT_PATH)
print("Output:", OUTPUT_DIR)


Input: C:\Landslide_FYP\Landslide_FYP_Development\04_model_ready\static_model_1to2.csv
Output: C:\Landslide_FYP\Landslide_FYP_Development\05_model_results\baseline_static


## 2. Load and validate model-ready dataset

In [6]:
df = pd.read_csv(INPUT_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))

display(
    df["target"]
    .value_counts()
    .sort_index()
    .rename_axis("target")
    .reset_index(name="records")
)

print(
    "Spatial blocks:",
    df["spatial_block_id"].nunique()
)

if df["target"].isna().any():
    raise ValueError("Missing target values found.")

if df["spatial_block_id"].isna().any():
    raise ValueError("Missing spatial block IDs found.")


Rows: 582
Columns: 53


,target,records
0,0,388
1,1,194


Spatial blocks: 167


## 3. Define predictors

The first baseline uses only the agreed core static variables:

- elevation
- slope
- aspect sine
- aspect cosine
- clay
- sand
- silt
- bulk density
- distance to major fault

No land cover, rainfall, road, waterway, or geology category is added yet.


In [7]:
FEATURES = [
    "elevation_m",
    "slope_deg",
    "aspect_sin",
    "aspect_cos",
    "soil_clay",
    "soil_sand",
    "soil_silt",
    "soil_bulk_density",
    "distance_major_fault_m",
]

TARGET = "target"
GROUP = "spatial_block_id"

missing_features = [
    c for c in FEATURES
    if c not in df.columns
]

if missing_features:
    raise KeyError(
        f"Missing predictors: {missing_features}"
    )

X = df[FEATURES].copy()
y = df[TARGET].astype(int).copy()
groups = df[GROUP].astype(str).copy()

print("Predictors:")
for feature in FEATURES:
    print(" -", feature)

print("\nMissing values:")
display(
    X.isna()
    .sum()
    .rename("missing")
    .reset_index()
    .rename(columns={"index": "feature"})
)


Predictors:
 - elevation_m
 - slope_deg
 - aspect_sin
 - aspect_cos
 - soil_clay
 - soil_sand
 - soil_silt
 - soil_bulk_density
 - distance_major_fault_m

Missing values:


,feature,missing
0,elevation_m,0
1,slope_deg,0
2,aspect_sin,0
3,aspect_cos,0
4,soil_clay,0
5,soil_sand,0
6,soil_silt,0
7,soil_bulk_density,0
8,distance_major_fault_m,0


## 4. Create 5 spatial validation folds

Preferred splitter:

`StratifiedGroupKFold`

This tries to preserve class balance while ensuring that the same spatial block never appears in both train and test data.

If the installed scikit-learn version cannot use it, the notebook falls back to `GroupKFold`.


In [8]:
N_SPLITS = 5
RANDOM_SEED = 42

try:
    splitter = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_SEED
    )

    splits = list(
        splitter.split(
            X,
            y,
            groups
        )
    )

    SPLITTER_NAME = "StratifiedGroupKFold"

except Exception as exc:

    print(
        "StratifiedGroupKFold unavailable/failed:",
        exc
    )

    splitter = GroupKFold(
        n_splits=N_SPLITS
    )

    splits = list(
        splitter.split(
            X,
            y,
            groups
        )
    )

    SPLITTER_NAME = "GroupKFold"


print(
    "Validation design:",
    SPLITTER_NAME
)

print(
    "Number of folds:",
    len(splits)
)


Validation design: StratifiedGroupKFold
Number of folds: 5


## 5. Inspect fold composition

In [9]:
fold_rows = []

for fold, (train_idx, test_idx) in enumerate(
    splits,
    start=1
):

    train_y = y.iloc[train_idx]
    test_y = y.iloc[test_idx]

    train_groups = set(
        groups.iloc[train_idx]
    )

    test_groups = set(
        groups.iloc[test_idx]
    )

    overlap = (
        train_groups
        .intersection(
            test_groups
        )
    )

    fold_rows.append({
        "fold": fold,
        "train_rows": len(train_idx),
        "test_rows": len(test_idx),
        "train_positive": int((train_y == 1).sum()),
        "train_control": int((train_y == 0).sum()),
        "test_positive": int((test_y == 1).sum()),
        "test_control": int((test_y == 0).sum()),
        "train_blocks": len(train_groups),
        "test_blocks": len(test_groups),
        "group_overlap": len(overlap),
    })

fold_summary = pd.DataFrame(
    fold_rows
)

display(fold_summary)

if fold_summary["group_overlap"].sum() != 0:
    raise ValueError(
        "Spatial leakage detected between train/test groups."
    )

if (fold_summary["test_positive"] == 0).any():
    raise ValueError(
        "At least one test fold has no positive landslides."
    )

if (fold_summary["test_control"] == 0).any():
    raise ValueError(
        "At least one test fold has no controls."
    )

fold_summary.to_csv(
    OUTPUT_DIR / "spatial_cv_fold_summary.csv",
    index=False
)


,fold,train_rows,test_rows,train_positive,train_control,test_positive,test_control,train_blocks,test_blocks,group_overlap
0,1,464,118,154,310,40,78,132,35,0
1,2,463,119,154,309,40,79,134,33,0
2,3,467,115,156,311,38,77,135,32,0
3,4,467,115,156,311,38,77,132,35,0
4,5,467,115,156,311,38,77,135,32,0


# PART A — MODEL DEFINITIONS

## 6. Logistic Regression

Numerical variables are standardized **inside each training fold**.

This avoids leakage.


In [10]:
numeric_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_preprocessor,
            FEATURES
        )
    ],
    remainder="drop"
)


logistic_model = Pipeline(
    steps=[
        (
            "preprocess",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                random_state=RANDOM_SEED
            )
        ),
    ]
)

print(logistic_model)


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['elevation_m', 'slope_deg',
                                                   'aspect_sin', 'aspect_cos',
                                                   'soil_clay', 'soil_sand',
                                                   'soil_silt',
                                                   'soil_bulk_density',
                                                   'distance_major_fault_m'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=5000,
        

## 7. Random Forest

The first Random Forest is intentionally conservative rather than heavily tuned.

Hyperparameter optimization comes later only if the baseline is stable.


In [11]:
rf_model = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=None,
                min_samples_leaf=3,
                max_features="sqrt",
                class_weight="balanced",
                random_state=RANDOM_SEED,
                n_jobs=-1
            )
        ),
    ]
)

print(rf_model)


Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('model',
                 RandomForestClassifier(class_weight='balanced',
                                        min_samples_leaf=3, n_estimators=500,
                                        n_jobs=-1, random_state=42))])


# PART B — SHARED SPATIAL CROSS-VALIDATION

## 8. Evaluation helper


In [12]:
def evaluate_model_spatial_cv(
    model_name,
    estimator,
    X,
    y,
    groups,
    splits
):

    oof_probability = np.full(
        len(y),
        np.nan,
        dtype=float
    )

    oof_prediction = np.full(
        len(y),
        np.nan,
        dtype=float
    )

    fold_metrics = []

    fitted_models = []


    for fold, (train_idx, test_idx) in enumerate(
        splits,
        start=1
    ):

        model = clone(
            estimator
        )

        X_train = X.iloc[
            train_idx
        ].copy()

        X_test = X.iloc[
            test_idx
        ].copy()

        y_train = y.iloc[
            train_idx
        ].copy()

        y_test = y.iloc[
            test_idx
        ].copy()


        model.fit(
            X_train,
            y_train
        )


        probability = model.predict_proba(
            X_test
        )[:, 1]


        prediction = (
            probability >= 0.5
        ).astype(int)


        oof_probability[
            test_idx
        ] = probability

        oof_prediction[
            test_idx
        ] = prediction


        metrics = {
            "model":
                model_name,

            "fold":
                fold,

            "accuracy":
                accuracy_score(
                    y_test,
                    prediction
                ),

            "balanced_accuracy":
                balanced_accuracy_score(
                    y_test,
                    prediction
                ),

            "precision":
                precision_score(
                    y_test,
                    prediction,
                    zero_division=0
                ),

            "recall":
                recall_score(
                    y_test,
                    prediction,
                    zero_division=0
                ),

            "f1":
                f1_score(
                    y_test,
                    prediction,
                    zero_division=0
                ),

            "roc_auc":
                roc_auc_score(
                    y_test,
                    probability
                ),

            "pr_auc":
                average_precision_score(
                    y_test,
                    probability
                ),

            "test_rows":
                len(
                    test_idx
                ),

            "test_positive":
                int(
                    y_test.sum()
                ),

            "test_controls":
                int(
                    (
                        y_test == 0
                    ).sum()
                ),
        }


        fold_metrics.append(
            metrics
        )

        fitted_models.append(
            model
        )


        print(
            model_name,
            "| fold",
            fold,
            "| ROC-AUC:",
            round(
                metrics[
                    "roc_auc"
                ],
                3
            ),
            "| balanced acc:",
            round(
                metrics[
                    "balanced_accuracy"
                ],
                3
            )
        )


    fold_metrics = pd.DataFrame(
        fold_metrics
    )


    overall_prediction = (
        oof_probability >= 0.5
    ).astype(int)


    overall = {
        "model":
            model_name,

        "accuracy":
            accuracy_score(
                y,
                overall_prediction
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y,
                overall_prediction
            ),

        "precision":
            precision_score(
                y,
                overall_prediction,
                zero_division=0
            ),

        "recall":
            recall_score(
                y,
                overall_prediction,
                zero_division=0
            ),

        "f1":
            f1_score(
                y,
                overall_prediction,
                zero_division=0
            ),

        "roc_auc":
            roc_auc_score(
                y,
                oof_probability
            ),

        "pr_auc":
            average_precision_score(
                y,
                oof_probability
            ),
    }


    return {
        "fold_metrics":
            fold_metrics,

        "overall":
            overall,

        "oof_probability":
            oof_probability,

        "oof_prediction":
            overall_prediction,

        "fitted_models":
            fitted_models,
    }


## 9. Train Logistic Regression

In [13]:
logistic_results = (
    evaluate_model_spatial_cv(
        "Logistic Regression",
        logistic_model,
        X,
        y,
        groups,
        splits
    )
)

display(
    logistic_results[
        "fold_metrics"
    ]
)


Logistic Regression | fold 1 | ROC-AUC: 0.777 | balanced acc: 0.715
Logistic Regression | fold 2 | ROC-AUC: 0.319 | balanced acc: 0.366
Logistic Regression | fold 3 | ROC-AUC: 0.764 | balanced acc: 0.68
Logistic Regression | fold 4 | ROC-AUC: 0.402 | balanced acc: 0.444
Logistic Regression | fold 5 | ROC-AUC: 0.524 | balanced acc: 0.515


,model,fold,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc,test_rows,test_positive,test_controls
0,Logistic Regression,1,0.703390,0.714744,0.545455,0.750000,0.631579,0.777244,0.662020,118,40,78
1,Logistic Regression,2,0.453782,0.366456,0.121212,0.100000,0.109589,0.318671,0.250380,119,40,79
2,Logistic Regression,3,0.678261,0.679768,0.509804,0.684211,0.584270,0.763841,0.630477,115,38,77
3,Logistic Regression,4,0.469565,0.443951,0.274510,0.368421,0.314607,0.401572,0.337329,115,38,77
4,Logistic Regression,5,0.582609,0.515038,0.352941,0.315789,0.333333,0.524265,0.345476,115,38,77


## 10. Train Random Forest

In [14]:
rf_results = (
    evaluate_model_spatial_cv(
        "Random Forest",
        rf_model,
        X,
        y,
        groups,
        splits
    )
)

display(
    rf_results[
        "fold_metrics"
    ]
)


Random Forest | fold 1 | ROC-AUC: 0.805 | balanced acc: 0.729
Random Forest | fold 2 | ROC-AUC: 0.553 | balanced acc: 0.505
Random Forest | fold 3 | ROC-AUC: 0.801 | balanced acc: 0.725
Random Forest | fold 4 | ROC-AUC: 0.644 | balanced acc: 0.613
Random Forest | fold 5 | ROC-AUC: 0.758 | balanced acc: 0.691


,model,fold,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc,test_rows,test_positive,test_controls
0,Random Forest,1,0.762712,0.729167,0.657895,0.625000,0.641026,0.805128,0.698907,118,40,78
1,Random Forest,2,0.613445,0.505222,0.350000,0.175000,0.233333,0.553165,0.359047,119,40,79
2,Random Forest,3,0.765217,0.724710,0.657143,0.605263,0.630137,0.800752,0.755043,115,38,77
3,Random Forest,4,0.669565,0.613295,0.500000,0.447368,0.472222,0.643882,0.525862,115,38,77
4,Random Forest,5,0.765217,0.691388,0.720000,0.473684,0.571429,0.758031,0.675536,115,38,77


# PART C — COMPARE MODELS

## 11. Overall out-of-fold metrics


In [15]:
overall_metrics = pd.DataFrame(
    [
        logistic_results[
            "overall"
        ],
        rf_results[
            "overall"
        ],
    ]
)

display(
    overall_metrics
)

overall_metrics.to_csv(
    OUTPUT_DIR
    / "baseline_model_overall_metrics.csv",
    index=False
)


,model,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc
0,Logistic Regression,0.577320,0.543814,0.383929,0.443299,0.411483,0.526398,0.369680
1,Random Forest,0.714777,0.652062,0.592105,0.463918,0.520231,0.707302,0.578731


## 12. Fold mean ± standard deviation

A model that performs well in one area but badly in other spatial folds is less trustworthy.

Therefore consistency matters as much as the overall score.


In [16]:
all_fold_metrics = pd.concat(
    [
        logistic_results[
            "fold_metrics"
        ],
        rf_results[
            "fold_metrics"
        ],
    ],
    ignore_index=True
)

metric_cols = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
]


summary_rows = []

for model_name, group in all_fold_metrics.groupby(
    "model"
):

    row = {
        "model":
            model_name
    }

    for metric in metric_cols:

        row[
            f"{metric}_mean"
        ] = group[
            metric
        ].mean()

        row[
            f"{metric}_std"
        ] = group[
            metric
        ].std()

    summary_rows.append(
        row
    )


fold_metric_summary = pd.DataFrame(
    summary_rows
)

display(
    fold_metric_summary
)


all_fold_metrics.to_csv(
    OUTPUT_DIR
    / "baseline_model_fold_metrics.csv",
    index=False
)


fold_metric_summary.to_csv(
    OUTPUT_DIR
    / "baseline_model_fold_summary.csv",
    index=False
)


,model,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std
0,Logistic Regression,0.577521,0.115090,0.543991,0.149964,0.360784,0.174076,0.443684,0.270103,0.394676,0.214195,0.557119,0.208159,0.445136,0.187666
1,Random Forest,0.715231,0.070174,0.652756,0.094647,0.577008,0.150726,0.465263,0.180111,0.509629,0.168334,0.712192,0.110156,0.602879,0.160499


# PART D — CONFUSION MATRICES

## 13. Overall out-of-fold confusion matrices


In [17]:
confusion_rows = []


for model_name, result in {
    "Logistic Regression":
        logistic_results,

    "Random Forest":
        rf_results,
}.items():

    cm = confusion_matrix(
        y,
        result[
            "oof_prediction"
        ]
    )

    tn, fp, fn, tp = cm.ravel()

    confusion_rows.append({
        "model":
            model_name,

        "true_negative":
            int(
                tn
            ),

        "false_positive":
            int(
                fp
            ),

        "false_negative":
            int(
                fn
            ),

        "true_positive":
            int(
                tp
            ),
    })


confusion_summary = pd.DataFrame(
    confusion_rows
)

display(
    confusion_summary
)


confusion_summary.to_csv(
    OUTPUT_DIR
    / "baseline_confusion_matrices.csv",
    index=False
)


,model,true_negative,false_positive,false_negative,true_positive
0,Logistic Regression,250,138,108,86
1,Random Forest,326,62,104,90


## 14. Confusion-matrix figures


In [18]:
for model_name, result in {
    "Logistic Regression":
        logistic_results,

    "Random Forest":
        rf_results,
}.items():

    cm = confusion_matrix(
        y,
        result[
            "oof_prediction"
        ]
    )

    fig, ax = plt.subplots(
        figsize=(6, 5)
    )

    image = ax.imshow(
        cm
    )

    ax.set_xticks(
        [0, 1]
    )

    ax.set_yticks(
        [0, 1]
    )

    ax.set_xticklabels(
        [
            "Pseudo-absence",
            "Landslide"
        ]
    )

    ax.set_yticklabels(
        [
            "Pseudo-absence",
            "Landslide"
        ]
    )

    ax.set_xlabel(
        "Predicted"
    )

    ax.set_ylabel(
        "Actual"
    )

    ax.set_title(
        f"{model_name} — Spatial CV Confusion Matrix"
    )

    for i in range(2):
        for j in range(2):
            ax.text(
                j,
                i,
                str(
                    cm[
                        i,
                        j
                    ]
                ),
                ha="center",
                va="center"
            )

    plt.tight_layout()

    filename = (
        model_name.lower()
        .replace(
            " ",
            "_"
        )
        + "_confusion_matrix.png"
    )

    fig.savefig(
        OUTPUT_DIR
        / filename,
        dpi=180,
        bbox_inches="tight"
    )

    plt.close(fig)


# PART E — ROC AND PRECISION-RECALL CURVES

## 15. ROC curves


In [19]:
fig, ax = plt.subplots(
    figsize=(7, 6)
)


for model_name, result in {
    "Logistic Regression":
        logistic_results,

    "Random Forest":
        rf_results,
}.items():

    fpr, tpr, _ = roc_curve(
        y,
        result[
            "oof_probability"
        ]
    )

    auc_value = roc_auc_score(
        y,
        result[
            "oof_probability"
        ]
    )

    ax.plot(
        fpr,
        tpr,
        label=(
            f"{model_name} "
            f"(AUC={auc_value:.3f})"
        )
    )


ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random"
)

ax.set_xlabel(
    "False Positive Rate"
)

ax.set_ylabel(
    "True Positive Rate"
)

ax.set_title(
    "Spatial Cross-Validated ROC Curves"
)

ax.legend()

ax.grid(
    True,
    alpha=0.2
)

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR
    / "baseline_roc_curves.png",
    dpi=180,
    bbox_inches="tight"
)

plt.close(fig)


## 16. Precision–Recall curves

In [20]:
fig, ax = plt.subplots(
    figsize=(7, 6)
)


for model_name, result in {
    "Logistic Regression":
        logistic_results,

    "Random Forest":
        rf_results,
}.items():

    precision, recall, _ = (
        precision_recall_curve(
            y,
            result[
                "oof_probability"
            ]
        )
    )

    ap = average_precision_score(
        y,
        result[
            "oof_probability"
        ]
    )

    ax.plot(
        recall,
        precision,
        label=(
            f"{model_name} "
            f"(AP={ap:.3f})"
        )
    )


ax.set_xlabel(
    "Recall"
)

ax.set_ylabel(
    "Precision"
)

ax.set_title(
    "Spatial Cross-Validated Precision–Recall Curves"
)

ax.legend()

ax.grid(
    True,
    alpha=0.2
)

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR
    / "baseline_precision_recall_curves.png",
    dpi=180,
    bbox_inches="tight"
)

plt.close(fig)


# PART F — OUT-OF-FOLD PREDICTIONS

## 17. Save susceptibility scores for every training sample

These are predictions made only when each sample was in a held-out spatial fold.

They are useful for honest diagnostic plots and later susceptibility-level design.


In [21]:
oof = df[
    [
        "sample_id",
        "target",
        "model_class",
        "longitude",
        "latitude",
        "spatial_block_id",
    ]
].copy()


oof[
    "logistic_score"
] = logistic_results[
    "oof_probability"
]


oof[
    "logistic_prediction_0_5"
] = logistic_results[
    "oof_prediction"
]


oof[
    "rf_score"
] = rf_results[
    "oof_probability"
]


oof[
    "rf_prediction_0_5"
] = rf_results[
    "oof_prediction"
]


oof.to_csv(
    OUTPUT_DIR
    / "baseline_oof_predictions.csv",
    index=False
)


display(
    oof.head()
)


,sample_id,target,model_class,longitude,latitude,spatial_block_id,logistic_score,logistic_prediction_0_5,rf_score,rf_prediction_0_5
0,POS_0001,1,landslide,100.2770,5.3630,B_222_11,0.144933,0,0.333685,0
1,POS_0002,1,landslide,100.2676,5.4175,B_222_12,0.453148,0,0.066345,0
2,POS_0003,1,landslide,116.5754,5.9913,B_259_13,0.673480,1,0.470204,0
3,POS_0006,1,landslide,101.8881,3.3902,B_226_7,0.632023,1,0.517264,1
4,POS_0007,1,landslide,101.6870,3.0879,B_225_6,0.447042,0,0.507642,1


# PART G — INTERPRETABILITY

## 18. Fit Logistic Regression on the full dataset for coefficient inspection

This fit is **only for interpretation after cross-validation**.

Performance must still be reported from spatial CV, not this full-data fit.


In [22]:
logistic_full = clone(
    logistic_model
)

logistic_full.fit(
    X,
    y
)


coefficients = (
    logistic_full
    .named_steps[
        "model"
    ]
    .coef_[0]
)


logistic_coefficient_table = pd.DataFrame({
    "feature":
        FEATURES,

    "standardized_coefficient":
        coefficients,

    "absolute_coefficient":
        np.abs(
            coefficients
        ),
}).sort_values(
    "absolute_coefficient",
    ascending=False
)


display(
    logistic_coefficient_table
)


logistic_coefficient_table.to_csv(
    OUTPUT_DIR
    / "logistic_regression_coefficients.csv",
    index=False
)


,feature,standardized_coefficient,absolute_coefficient
6,soil_silt,-0.354243,0.354243
5,soil_sand,0.328128,0.328128
0,elevation_m,0.287386,0.287386
4,soil_clay,-0.225300,0.225300
8,distance_major_fault_m,-0.216235,0.216235
1,slope_deg,-0.156586,0.156586
2,aspect_sin,0.112858,0.112858
3,aspect_cos,-0.017815,0.017815
7,soil_bulk_density,0.006667,0.006667


## 19. Fit Random Forest on the full dataset for feature-importance inspection

Again, this is for interpretation only.

Performance remains the spatial cross-validation result.


In [23]:
rf_full = clone(
    rf_model
)

rf_full.fit(
    X,
    y
)


rf_importance = (
    rf_full
    .named_steps[
        "model"
    ]
    .feature_importances_
)


rf_importance_table = pd.DataFrame({
    "feature":
        FEATURES,

    "importance":
        rf_importance,
}).sort_values(
    "importance",
    ascending=False
)


display(
    rf_importance_table
)


rf_importance_table.to_csv(
    OUTPUT_DIR
    / "random_forest_feature_importance.csv",
    index=False
)


,feature,importance
7,soil_bulk_density,0.228560
0,elevation_m,0.168351
8,distance_major_fault_m,0.157419
5,soil_sand,0.107523
6,soil_silt,0.091052
4,soil_clay,0.084313
1,slope_deg,0.056261
3,aspect_cos,0.055221
2,aspect_sin,0.051300


# PART H — SUSCEPTIBILITY LEVELS

## 20. Create provisional relative susceptibility classes

Because this is a pseudo-absence case-control design, the raw scores should not automatically be called literal future probabilities.

For the FYP dashboard, a defensible first approach is to transform continuous scores into **relative susceptibility classes** based on score quantiles.

Example 5-class system:

- Very Low
- Low
- Moderate
- High
- Very High

The thresholds below are based on the **Random Forest out-of-fold score distribution**.

These are provisional and should later be finalized after model selection and Malaysia-wide prediction.


In [24]:
rf_scores = oof[
    "rf_score"
].dropna()


quantile_thresholds = rf_scores.quantile(
    [
        0.20,
        0.40,
        0.60,
        0.80,
    ]
)


q20 = float(
    quantile_thresholds.loc[
        0.20
    ]
)

q40 = float(
    quantile_thresholds.loc[
        0.40
    ]
)

q60 = float(
    quantile_thresholds.loc[
        0.60
    ]
)

q80 = float(
    quantile_thresholds.loc[
        0.80
    ]
)


def susceptibility_class(
    score
):

    if score <= q20:
        return "Very Low"

    if score <= q40:
        return "Low"

    if score <= q60:
        return "Moderate"

    if score <= q80:
        return "High"

    return "Very High"


oof[
    "rf_relative_susceptibility_class"
] = oof[
    "rf_score"
].apply(
    susceptibility_class
)


threshold_table = pd.DataFrame({
    "boundary": [
        "Very Low / Low",
        "Low / Moderate",
        "Moderate / High",
        "High / Very High",
    ],

    "rf_score_threshold": [
        q20,
        q40,
        q60,
        q80,
    ]
})


display(
    threshold_table
)


threshold_table.to_csv(
    OUTPUT_DIR
    / "provisional_susceptibility_thresholds.csv",
    index=False
)


oof.to_csv(
    OUTPUT_DIR
    / "baseline_oof_predictions_with_classes.csv",
    index=False
)


,boundary,rf_score_threshold
0,Very Low / Low,0.167122
1,Low / Moderate,0.266953
2,Moderate / High,0.378901
3,High / Very High,0.554876


# How to interpret the result

## Static model

This model answers:

> **Which locations have environmental conditions similar to known landslide locations?**

It estimates **relative landslide susceptibility**, not consequence severity.

It does **not** directly predict:

- how many people will be affected
- whether a road will definitely be blocked
- economic loss
- building damage

Those are **exposure / consequence / risk-impact** questions.

---

# Road-blockage example

A future dashboard can combine layers:

```text
Landslide susceptibility map
        +
Road network
        ↓
Road segments intersecting High / Very High susceptibility zones
        ↓
Potential road-exposure / disruption warning
```

That is valid.

But the model should not claim:

> "this road will be blocked"

unless a separate consequence/road-disruption model is developed with actual road-blockage labels.

A defensible dashboard label is:

> **Road segment exposed to high landslide susceptibility**

---

# Semi-dynamic model

Later, temporal models can add:

- recent rainfall
- soil moisture
- historical/current land cover
- road context where defensible

This can support a **semi-dynamic landslide risk/susceptibility indicator**.

The final system can therefore show:

1. **Static susceptibility**
2. **Recent dynamic conditions**
3. **Updated relative risk indicator**
4. **Infrastructure exposure overlays**, such as roads


# Stage 07 outputs to review

Please send:

```text
spatial_cv_fold_summary.csv
baseline_model_overall_metrics.csv
baseline_model_fold_summary.csv
baseline_confusion_matrices.csv
logistic_regression_coefficients.csv
random_forest_feature_importance.csv
provisional_susceptibility_thresholds.csv
```

And preferably screenshots of:

```text
baseline_roc_curves.png
baseline_precision_recall_curves.png
```

After reviewing these, we can decide:

- which baseline model is stronger
- whether the performance is stable spatially
- whether to tune Random Forest
- whether XGBoost is worth adding
- how to finalize susceptibility levels for the dashboard
